In [1]:
import sys
import ray
import numpy as np
import tensorflow as tf
import gym

sys.path.append(r'C:\Users\adria\coding\katja\DRL-in-international-economy-ai-economist-')

from ai_economist import foundation

from utils import plotting

import ray
from ray.rllib.agents.ppo import PPOTrainer

from rllib.env_wrapper import RLlibEnvWrapper


c:\Users\adria\anaconda3\envs\ai-econ-fixed\lib\site-packages\tensorflow\python\framework\dtypes.py:516: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
c:\Users\adria\anaconda3\envs\ai-econ-fixed\lib\site-packages\tensorflow\python\framework\dtypes.py:517: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
c:\Users\adria\anaconda3\envs\ai-econ-fixed\lib\site-packages\tensorflow\python\framework\dtypes.py:518: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
c:\Users\adria\anaconda3\envs\ai-econ-fixed\lib\s

In [2]:
# =========================
# Two-phase RLlib training + dual-mode rollout logging
# =========================
import os
import time
import gc
import numpy as np
import ray

from ray.rllib.agents.ppo import PPOTrainer
from tutorials.rllib.env_wrapper import RLlibEnvWrapper

# ----------------------------
# SETTINGS (adjust as you wish)
# ----------------------------
FRAMEWORK      = "tf"      # "tf" or "torch"
PHASE1_ITERS   = 20       # agents warm-up
PHASE2_ITERS   = 50       # planners learn
PERIOD         = 100       # tax decision period
EPISODE_LENGTH = 1000
WORLD_SIZE     = [100, 50]
LAYOUT_FILE    = "map_100x50_water_gaps_3percent_resources.txt"

# ----------------------------
# Ray init (Windows-safe)
# ----------------------------
ray.shutdown()
ray.init(ignore_reinit_error=True, log_to_driver=False)

# ----------------------------
# Policy mapping
# ----------------------------
def policy_mapping_fun(agent_id):
    aid = str(agent_id)
    if aid.isdigit():
        return "a"
    if aid == "p_top":
        return "p_top"
    if aid == "p_bottom":
        return "p_bottom"
    return "a"

# ----------------------------
# Env Configs (Phase 1 & 2)
# ----------------------------
env_config_dict_phase1 = {
    "scenario_name": "custom/splitworld_overlay_regional",
    "components": [
        ('Build', {
            'skill_dist': 'pareto',
            'payment_max_skill_multiplier': 3,
            'build_labor': 10,
            'payment': 10
        }),
        ('ContinuousDoubleAuction', {
            'max_bid_ask': 10,
            'order_labor': 0.25,
            'max_num_orders': 5,
            'order_duration': 50
        }),
        ('Gather', {
            'move_labor': 1,
            'collect_labor': 1,
            'skill_dist': 'pareto'
        }),
        ("RegionalPeriodicBracketTax", {
            "region": "top",
            "planner_id": "p_top",
            "period": PERIOD,
            "bracket_spacing": "us-federal",
            "usd_scaling": 1000,
            "disable_taxes": True,
        }),
        ("RegionalPeriodicBracketTax", {
            "region": "bottom",
            "planner_id": "p_bottom",
            "period": PERIOD,
            "bracket_spacing": "us-federal",
            "usd_scaling": 1000,
            "disable_taxes": True,
        }),
    ],
    "env_layout_file": LAYOUT_FILE,
    "world_size": WORLD_SIZE,
    "episode_length": EPISODE_LENGTH,
    "starting_agent_coin": 10,
    "fixed_four_skill_and_loc": False,
    "n_agents": 4,
    "planner_subclasses": ["TopPlanner", "BottomPlanner"],
    "multi_action_mode_planner": True,
    "multi_action_mode_agents": True,
    "flatten_observations": True,
    "flatten_masks": True,
    "dense_log_frequency": 1
}

env_config_dict_phase2 = {
    "scenario_name": "custom/splitworld_overlay_regional",
    "components": [
        ('Build', {
            'skill_dist': 'pareto',
            'payment_max_skill_multiplier': 3,
            'build_labor': 10,
            'payment': 10
        }),
        ('ContinuousDoubleAuction', {
            'max_bid_ask': 10,
            'order_labor': 0.25,
            'max_num_orders': 5,
            'order_duration': 50
        }),
        ('Gather', {
            'move_labor': 1,
            'collect_labor': 1,
            'skill_dist': 'pareto'
        }),
        ("RegionalPeriodicBracketTax", {
            "region": "top",
            "planner_id": "p_top",
            "period": PERIOD,
            "bracket_spacing": "us-federal",
            "usd_scaling": 1000,
            "disable_taxes": False,
            "tax_model": "model_wrapper",
            "tax_annealing_schedule": [-100, 0.001],
        }),
        ("RegionalPeriodicBracketTax", {
            "region": "bottom",
            "planner_id": "p_bottom",
            "period": PERIOD,
            "bracket_spacing": "us-federal",
            "usd_scaling": 1000,
            "disable_taxes": False,
            "tax_model": "model_wrapper",
            "tax_annealing_schedule": [-100, 0.001],
        }),
    ],
    "env_layout_file": LAYOUT_FILE,
    "world_size": WORLD_SIZE,
    "episode_length": EPISODE_LENGTH,
    "starting_agent_coin": 10,
    "fixed_four_skill_and_loc": False,
    "n_agents": 4,
    "planner_subclasses": ["TopPlanner", "BottomPlanner"],
    "multi_action_mode_planner": True,
    "multi_action_mode_agents": True,
    "flatten_observations": True,
    "flatten_masks": True,
    "dense_log_frequency": 1
}

# ----------------------------
# Build policies (add planner entropy)
# ----------------------------
def build_policies(env_obj):
    obs_space_a   = env_obj.observation_space
    act_space_a   = env_obj.action_space
    obs_space_top = env_obj.observation_space_pl["p_top"]
    act_space_top = env_obj.action_space_pl["p_top"]
    obs_space_bot = env_obj.observation_space_pl["p_bottom"]
    act_space_bot = env_obj.action_space_pl["p_bottom"]

    return {
        "a": (
            None, obs_space_a, act_space_a,
            {"lr": 3e-4}
        ),
        "p_top": (
            None, obs_space_top, act_space_top,
            {"lr": 1e-4, "entropy_coeff": 0.02}
        ),
        "p_bottom": (
            None, obs_space_bot, act_space_bot,
            {"lr": 1e-4, "entropy_coeff": 0.02}
        ),
    }

# ----------------------------
# PPO configs
# ----------------------------
def build_trainer_config(env_config_dict, policies, policies_to_train):
    return {
        "env": RLlibEnvWrapper,
        "env_config": {
            "env_config_dict": env_config_dict,
            "num_envs_per_worker": 1,
        },
        "multiagent": {
            "policies": policies,
            "policies_to_train": policies_to_train,
            "policy_mapping_fn": policy_mapping_fun,
        },
        "num_workers": 0,
        "num_envs_per_worker": 1,
        "framework": FRAMEWORK,
        "num_gpus": 0,

        "rollout_fragment_length": 50,
        "batch_mode": "truncate_episodes",
        "train_batch_size": 800,
        "sgd_minibatch_size": 128,
        "num_sgd_iter": 2,
        "log_level": "WARN",
    }

# ----------------------------
# Train loop
# ----------------------------
def train_phase(trainer, name, iters):
    start = time.time()
    for i in range(iters):
        r = trainer.train()
        if i % 25 == 0:
            print(f"[{name}] Iter={i:04d}/{iters} reward={r.get('episode_reward_mean')}")
        if i % 100 == 0 and i > 0:
            _ = trainer.save()
        if i % 10 == 0:
            gc.collect()
    ckpt = trainer.save()
    print(f"[{name}] Final checkpoint: {ckpt}  ({(time.time()-start)/60.0:.1f} min)")
    return ckpt

# ----------------------------
# Detect true tax decision day
# ----------------------------
def is_tax_day(env_obj):
    for c in env_obj.env.components:
        if "BracketTax" in c.name and getattr(c, "tax_cycle_pos", None) == 1:
            return True
    return False

# ----------------------------
# Rollout + dual-mode logging
#   mode: "all" (every step) or "decision" (only tax days)
# ----------------------------
def rollout_with_planner_actions(trainer, env_obj, episodes=2, mode="decision", explore=False):
    assert mode in ("all", "decision")
    def _compute_action(pid, obs, state):
        out = trainer.compute_action(
            observation=obs, state=state, policy_id=pid,
            full_fetch=False, explore=explore
        )
        if isinstance(out, tuple):
            if len(out) >= 2:
                return out[0], out[1]
            else:
                return out[0], state
        return out, state

    dense_logs = {}

    for ep in range(episodes):
        obs = env_obj.reset(force_dense_logging=True)

        agent_states = {
            str(i): trainer.get_policy("a").get_initial_state()
            for i in range(env_obj.env.n_agents)
        }
        p_top_state    = trainer.get_policy("p_top").get_initial_state()
        p_bottom_state = trainer.get_policy("p_bottom").get_initial_state()

        top_actions_all, bot_actions_all = [], []
        top_rewards, bot_rewards = [], []

        for _t in range(env_obj.env.episode_length):
            actions = {}
            for i in range(env_obj.env.n_agents):
                aid = str(i)
                a, ns = _compute_action("a", obs[aid], agent_states[aid])
                actions[aid] = a
                agent_states[aid] = ns

            a_top,    p_top_state    = _compute_action("p_top",    obs["p_top"],    p_top_state)
            a_bottom, p_bottom_state = _compute_action("p_bottom", obs["p_bottom"], p_bottom_state)
            actions["p_top"]    = a_top
            actions["p_bottom"] = a_bottom

            # Logging
            if mode == "all" or (mode == "decision" and is_tax_day(env_obj)):
                top_actions_all.append(np.array(a_top, copy=True))
                bot_actions_all.append(np.array(a_bottom, copy=True))

            obs, rew, done, info = env_obj.step(actions)

            top_rewards.append(rew.get("p_top", np.nan))
            bot_rewards.append(rew.get("p_bottom", np.nan))
            if done.get("__all__", False):
                break

        # Stack logs (deep-copied per step)
        top_arr = np.stack(top_actions_all, axis=0) if len(top_actions_all) else np.zeros((0, 0), dtype=np.int64)
        bot_arr = np.stack(bot_actions_all, axis=0) if len(bot_actions_all) else np.zeros((0, 0), dtype=np.int64)

        dense_logs[ep] = dict(env_obj.env.dense_log)
        dense_logs[ep]["planner_actions"] = {"p_top": top_arr, "p_bottom": bot_arr}
        dense_logs[ep]["planner_rewards"] = {"p_top": top_rewards, "p_bottom": bot_rewards}

    return dense_logs

# =========================================================
# PHASE 1 — Agents only
# =========================================================
env_obj_phase1 = RLlibEnvWrapper({"env_config_dict": env_config_dict_phase1}, verbose=False)
policies_phase1 = build_policies(env_obj_phase1)

trainer_phase1 = PPOTrainer(
    config=build_trainer_config(env_config_dict_phase1, policies_phase1, policies_to_train=["a"])
)
_ = train_phase(trainer_phase1, "PHASE 1", PHASE1_ITERS)

# Copy trained agent weights to Phase 2
agent_weights = trainer_phase1.get_policy("a").get_weights()
trainer_phase1.stop()
del trainer_phase1
gc.collect()

# =========================================================
# PHASE 2 — Planners only
# =========================================================
env_obj_phase2 = RLlibEnvWrapper({"env_config_dict": env_config_dict_phase2}, verbose=False)
policies_phase2 = build_policies(env_obj_phase2)

trainer_phase2 = PPOTrainer(
    config=build_trainer_config(env_config_dict_phase2, policies_phase2, policies_to_train=["p_top","p_bottom"])
)
trainer_phase2.get_policy("a").set_weights(agent_weights)

_ = train_phase(trainer_phase2, "PHASE 2", PHASE2_ITERS)

# =========================================================
# Rollout in BOTH MODES
# =========================================================
dense_logs_decision = rollout_with_planner_actions(
    trainer=trainer_phase2, env_obj=env_obj_phase2,
    episodes=1, mode="decision", explore=False
)
dense_logs_all = rollout_with_planner_actions(
    trainer=trainer_phase2, env_obj=env_obj_phase2,
    episodes=1, mode="all", explore=False
)

# Sanity: unique counts (should be > 1)
arr_top_dec = dense_logs_decision[0]["planner_actions"]["p_top"]
arr_bot_dec = dense_logs_decision[0]["planner_actions"]["p_bottom"]
print("[DECISION] p_top:", arr_top_dec.shape, "unique:", np.unique(arr_top_dec, axis=0).shape[0] if arr_top_dec.size else 0)
print("[DECISION] p_bottom:", arr_bot_dec.shape, "unique:", np.unique(arr_bot_dec, axis=0).shape[0] if arr_bot_dec.size else 0)

arr_top_all = dense_logs_all[0]["planner_actions"]["p_top"]
arr_bot_all = dense_logs_all[0]["planner_actions"]["p_bottom"]
print("[ALL] p_top:", arr_top_all.shape, "unique:", np.unique(arr_top_all, axis=0).shape[0] if arr_top_all.size else 0)
print("[ALL] p_bottom:", arr_bot_all.shape, "unique:", np.unique(arr_bot_all, axis=0).shape[0] if arr_bot_all.size else 0)

2026-03-19 17:15:24,022	INFO resource_spec.py:212 -- Starting Ray with 3.91 GiB memory available for workers and up to 1.97 GiB for objects. You can adjust these settings with ray.init(memory=<bytes>, object_store_memory=<bytes>).
2026-03-19 17:15:24,440	INFO services.py:1165 -- View the Ray dashboard at localhost:8265
2026-03-19 17:15:29,646	ERROR syncer.py:46 -- Log sync requires rsync to be installed.
2026-03-19 17:15:29,648	INFO trainer.py:585 -- Tip: set framework=tfe or the --eager flag to enable TensorFlow eager execution
2026-03-19 17:15:29,648	INFO trainer.py:612 -- Current log_level is WARN. For more information, set 'log_level': 'INFO' / 'DEBUG' or use the -v and -vv flags.


Instructions for updating:
Use `tf.random.categorical` instead.
Instructions for updating:
keep_dims is deprecated, use keepdims instead
Instructions for updating:
keep_dims is deprecated, use keepdims instead
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where


2026-03-19 17:15:52,257	INFO trainable.py:181 -- _setup took 22.610 seconds. If your trainable is slow to initialize, consider setting reuse_actors=True to reduce actor creation overheads.


Instructions for updating:
Prefer Variable.assign which has equivalent behavior in 2.X.
[PHASE 1] Iter=0000/20 reward=nan
[PHASE 1] Final checkpoint: C:\Users\adria/ray_results\PPO_RLlibEnvWrapper_2026-03-19_17-15-29ofgvi3qg\checkpoint_20\checkpoint-20  (2.4 min)


2026-03-19 17:18:18,076	ERROR syncer.py:46 -- Log sync requires rsync to be installed.
2026-03-19 17:19:04,931	INFO trainable.py:181 -- _setup took 46.853 seconds. If your trainable is slow to initialize, consider setting reuse_actors=True to reduce actor creation overheads.


[PHASE 2] Iter=0000/50 reward=nan
[PHASE 2] Iter=0025/50 reward=-328.38644559517877
[PHASE 2] Final checkpoint: C:\Users\adria/ray_results\PPO_RLlibEnvWrapper_2026-03-19_17-18-180d1rk5h8\checkpoint_50\checkpoint-50  (7.9 min)
[DECISION] p_top: (10, 14) unique: 1
[DECISION] p_bottom: (10, 14) unique: 2
[ALL] p_top: (1000, 14) unique: 2
[ALL] p_bottom: (1000, 14) unique: 4


In [3]:
import numpy as np
import matplotlib.pyplot as plt

def _get_disc_rates(env_obj):
    """Returns discrete marginal tax rate values (0..1)."""
    for comp in env_obj.env.components:
        if "BracketTax" in comp.name and hasattr(comp, "disc_rates"):
            return np.array(comp.disc_rates, dtype=float)
    return np.linspace(0, 1, 21)   # fallback if needed


def _split_top_bottom_matrix(A, top_first=True):
    """
    Split a (T,S) matrix into top7 and bottom7.
    Works with S=14 (top-first or bottom-first).
    """
    if A.size == 0:
        return A, A
    T, S = A.shape
    if S == 7:
        return A, A         # rare, but handle gracefully
    half = S // 2
    if top_first:
        return A[:, :half], A[:, half:]
    else:
        return A[:, half:], A[:, :half]
    

def plot_planner_tax_lines_percent(
    dense_log,
    env_obj,
    brackets,
    title="Planner Marginal Rate Choices (per decision)",
    top_first=True,
    figsize=(14, 10),
):
    """
    Plots p_top vs p_bottom marginal tax choices for each bracket over decisions.
    Expects dense_log["planner_actions"]["p_top"] to be (T,14) or (T,7) depending
    on your action setup.
    """
    actions_top  = dense_log["planner_actions"]["p_top"]     # (T,S)
    actions_bot  = dense_log["planner_actions"]["p_bottom"]  # (T,S)

    if actions_top.size == 0:
        print("No planner actions logged.")
        return None

    T = actions_top.shape[0]
    X = np.arange(T)   # decision index

    # Extract 7 indices for each planner
    top7_top, bottom7_top       = _split_top_bottom_matrix(actions_top, top_first)
    top7_bottom, bottom7_bottom = _split_top_bottom_matrix(actions_bot, top_first)

    # For timeline: p_top’s top7, p_bottom’s bottom7
    mat_top    = top7_top        # (T,7)
    mat_bottom = bottom7_bottom  # (T,7)

    disc_rates = _get_disc_rates(env_obj)
    # Clip indices to safety
    mat_top_r    = disc_rates[np.clip(mat_top,    0, len(disc_rates)-1)]
    mat_bottom_r = disc_rates[np.clip(mat_bottom, 0, len(disc_rates)-1)]

    fig, axes = plt.subplots(7, 1, figsize=figsize, sharex=True)
    fig.suptitle(title)

    for b in range(7):
        ax = axes[b]
        ax.plot(X, mat_top_r[:, b],    label="p_top",    lw=2, color="#1f77b4")
        ax.plot(X, mat_bottom_r[:, b], label="p_bottom", lw=2, color="#ff7f0e", ls="--")

        ax.set_ylabel(f"{brackets[b]:.2f}k")
        ax.grid(True, alpha=0.4)
        if b == 0:
            ax.legend()

    axes[-1].set_xlabel("Planner decision index (tax days)")
    plt.tight_layout()
    plt.show()
    return fig


In [4]:
def plot_final_tax_schedules_two_planners(
    dense_log,
    env_obj,
    brackets,
    title="Final Marginal Tax Schedules",
    top_first=True,
    figsize=(10, 7),
):
    """
    Produces two stacked step-plots:
       top = p_top final schedule
       bottom = p_bottom final schedule
    """
    actions_top  = dense_log["planner_actions"]["p_top"]
    actions_bot  = dense_log["planner_actions"]["p_bottom"]

    if actions_top.size == 0:
        print("No planner actions logged.")
        return None

    last_top    = actions_top[-1]
    last_bottom = actions_bot[-1]

    top7_top, bottom7_top           = _split_top_bottom_matrix(last_top.reshape(1,-1), top_first)
    top7_bottom, bottom7_bottom     = _split_top_bottom_matrix(last_bottom.reshape(1,-1), top_first)

    idx_top    = top7_top[0]         # (7,)
    idx_bottom = bottom7_bottom[0]   # (7,)

    disc_rates = _get_disc_rates(env_obj)

    top_rates    = disc_rates[np.clip(idx_top,    0, len(disc_rates)-1)]
    bottom_rates = disc_rates[np.clip(idx_bottom, 0, len(disc_rates)-1)]

    fig, axes = plt.subplots(2, 1, figsize=figsize, sharex=True)
    fig.suptitle(title)

    # p_top
    axes[0].step(brackets, top_rates, where="post", color="#1f77b4")
    axes[0].fill_between(brackets, top_rates, step="post", alpha=0.3, color="#1f77b4")
    axes[0].set_ylabel("Marginal Rate")
    axes[0].set_title("p_top (Top Region)")
    axes[0].grid(True)

    # p_bottom
    axes[1].step(brackets, bottom_rates, where="post", color="#ff7f0e")
    axes[1].fill_between(brackets, bottom_rates, step="post", alpha=0.3, color="#ff7f0e")
    axes[1].set_ylabel("Marginal Rate")
    axes[1].set_title("p_bottom (Bottom Region)")
    axes[1].set_xlabel("Income Bracket (k USD)")
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()
    return fig

In [5]:
dense_logs_decision = rollout_with_planner_actions(
    trainer=trainer_phase2,
    env_obj=env_obj_phase2,
    episodes=1,
    mode="decision",   # every tax day
)
dense_logs_all = rollout_with_planner_actions(
    trainer=trainer_phase2,
    env_obj=env_obj_phase2,
    episodes=1,
    mode="all",        # every step
)

In [6]:
plot_planner_tax_lines_percent(
    dense_logs_decision[0],
    env_obj_phase2,
    brackets,
    title="Planner Marginal Rates Over Decisions (decision-only)"
)

NameError: name 'brackets' is not defined

In [ ]:
# import os
# import time
# import gc
# import ray

# from ray.rllib.agents.ppo import PPOTrainer
# from ray.tune.logger import UnifiedLogger
# from tutorials.rllib.env_wrapper import RLlibEnvWrapper

# # =========================================================
# # Ray init
# # =========================================================
# ray.init(ignore_reinit_error=True, log_to_driver=False)

# # =========================================================
# # Output dirs
# # =========================================================
# PHASE1_DIR = "phase1_tb"
# PHASE2_DIR = "phase2_tb"
# os.makedirs(PHASE1_DIR, exist_ok=True)
# os.makedirs(PHASE2_DIR, exist_ok=True)

# def make_logger_creator(base_dir):
#     def logger_creator(config):
#         ts = time.strftime("%Y-%m-%d_%H-%M-%S")
#         logdir = os.path.join(base_dir, f"run-{ts}")
#         os.makedirs(logdir, exist_ok=True)
#         return UnifiedLogger(config, logdir, loggers=None)
#     return logger_creator

# def policy_mapping_fun(agent_id):
#     aid = str(agent_id)
#     if aid.isdigit():
#         return "a"
#     if aid == "p_top":
#         return "p_top"
#     if aid == "p_bottom":
#         return "p_bottom"
#     return "a"

# # =========================================================
# # Phase 1 env config
# # =========================================================
# env_config_dict_phase1 = {
#     "scenario_name": "custom/splitworld_overlay_regional",
#     "components": [
#         ('Build', {
#             'skill_dist': 'pareto',
#             'payment_max_skill_multiplier': 3,
#             'build_labor': 10,
#             'payment': 10
#         }),
#         ('ContinuousDoubleAuction', {
#             'max_bid_ask': 10,
#             'order_labor': 0.25,
#             'max_num_orders': 5,
#             'order_duration': 50
#         }),
#         ('Gather', {
#             'move_labor': 1,
#             'collect_labor': 1,
#             'skill_dist': 'pareto'
#         }),
#         ("RegionalPeriodicBracketTax", {
#             "region": "top",
#             "planner_id": "p_top",
#             "period": 100,
#             "bracket_spacing": "us-federal",
#             "usd_scaling": 1000,
#             "disable_taxes": True,
#         }),
#         ("RegionalPeriodicBracketTax", {
#             "region": "bottom",
#             "planner_id": "p_bottom",
#             "period": 100,
#             "bracket_spacing": "us-federal",
#             "usd_scaling": 1000,
#             "disable_taxes": True,
#         }),
#     ],
#     "env_layout_file": "map_100x50_water_gaps_3percent_resources.txt",
#     "world_size": [100, 50],
#     "episode_length": 1000,
#     "starting_agent_coin": 10,
#     "fixed_four_skill_and_loc": False,
#     "n_agents": 4,
#     "planner_subclasses": ["TopPlanner", "BottomPlanner"],
#     "multi_action_mode_planner": True,
#     "multi_action_mode_agents": True,
#     "flatten_observations": True,
#     "flatten_masks": True,
#     "dense_log_frequency": 1
# }

# # Build once to capture spaces
# env_obj_phase1 = RLlibEnvWrapper({"env_config_dict": env_config_dict_phase1}, verbose=False)

# obs_space_a   = env_obj_phase1.observation_space
# act_space_a   = env_obj_phase1.action_space
# obs_space_top = env_obj_phase1.observation_space_pl["p_top"]
# act_space_top = env_obj_phase1.action_space_pl["p_top"]
# obs_space_bot = env_obj_phase1.observation_space_pl["p_bottom"]
# act_space_bot = env_obj_phase1.action_space_pl["p_bottom"]

# policies = {
#     "a": (
#         None, obs_space_a, act_space_a,
#         {"lr": 3e-4}
#     ),
#     "p_top": (
#         None, obs_space_top, act_space_top,
#         {"lr": 1e-4}
#     ),
#     "p_bottom": (
#         None, obs_space_bot, act_space_bot,
#         {"lr": 1e-4}
#     ),
# }

# # =========================================================
# # Common safer config
# # =========================================================
# def build_trainer_config(env_config_dict, policies_to_train):
#     return {
#         "env": RLlibEnvWrapper,
#         "env_config": {
#             "env_config_dict": env_config_dict,
#             "num_envs_per_worker": 1,
#         },
#         "multiagent": {
#             "policies": policies,
#             "policies_to_train": policies_to_train,
#             "policy_mapping_fn": policy_mapping_fun,
#         },
#         "num_workers": 0,
#         "num_envs_per_worker": 1,
#         "framework": "tf",
#         "num_gpus": 0,

#         # Safer memory settings
#         "rollout_fragment_length": 50,
#         "batch_mode": "truncate_episodes",
#         "train_batch_size": 500,
#         "sgd_minibatch_size": 64,
#         "num_sgd_iter": 2,

#         "log_level": "WARN",
#     }

# def run_phase(name, env_config_dict, policies_to_train, log_dir, iters, restore_path=None):
#     trainer = PPOTrainer(
#         config=build_trainer_config(env_config_dict, policies_to_train),
#         logger_creator=make_logger_creator(log_dir),
#     )

#     if restore_path is not None:
#         trainer.restore(restore_path)
#         print(f"[{name}] Restored from {restore_path}")

#     ckpt_path = None
#     start_time = time.time()

#     for i in range(iters):
#         result = trainer.train()

#         if i % 25 == 0:
#             elapsed_min = (time.time() - start_time) / 60.0
#             print(
#                 f"[{name}] Iter={i:04d}/{iters} "
#                 f"reward={result.get('episode_reward_mean')} "
#                 f"elapsed={elapsed_min:.1f} min"
#             )

#         if i % 100 == 0 and i > 0:
#             ckpt_path = trainer.save(log_dir)
#             print(f"[{name}] Saved checkpoint: {ckpt_path}")

#         if i % 10 == 0:
#             gc.collect()

#     ckpt_path = trainer.save(log_dir)
#     total_min = (time.time() - start_time) / 60.0
#     print(f"[{name}] Final checkpoint: {ckpt_path}")
#     print(f"[{name}] Finished in {total_min:.1f} minutes")

#     # Optional annealing verification
#     try:
#         for comp in trainer.workers.local_worker().env.env.components:
#             if "BracketTax" in comp.name:
#                 print(
#                     f"[{name}] {comp.name} "
#                     f"schedule={getattr(comp, 'tax_annealing_schedule', None)} "
#                     f"annealed_max={getattr(comp, '_annealed_rate_max', None)}"
#                 )
#     except Exception as e:
#         print(f"[{name}] Could not inspect tax components: {e}")

#     trainer.stop()
#     del trainer
#     gc.collect()
#     return ckpt_path

# # =========================================================
# # Run Phase 1
# # =========================================================
# PHASE1_ITERS = 200

# ckpt_phase1_path = run_phase(
#     name="PHASE 1",
#     env_config_dict=env_config_dict_phase1,
#     policies_to_train=["a"],
#     log_dir=PHASE1_DIR,
#     iters=PHASE1_ITERS,
# )

# # =========================================================
# # Phase 2 env config
# # =========================================================
# env_config_dict_phase2 = {
#     "scenario_name": "custom/splitworld_overlay_regional",
#     "components": [
#         ('Build', {
#             'skill_dist': 'pareto',
#             'payment_max_skill_multiplier': 3,
#             'build_labor': 10,
#             'payment': 10
#         }),
#         ('ContinuousDoubleAuction', {
#             'max_bid_ask': 10,
#             'order_labor': 0.25,
#             'max_num_orders': 5,
#             'order_duration': 50
#         }),
#         ('Gather', {
#             'move_labor': 1,
#             'collect_labor': 1,
#             'skill_dist': 'pareto'
#         }),
#         ("RegionalPeriodicBracketTax", {
#             "region": "top",
#             "planner_id": "p_top",
#             "period": 100,
#             "bracket_spacing": "us-federal",
#             "usd_scaling": 1000,
#             "disable_taxes": False,
#             "tax_model": "model_wrapper",
#             "tax_annealing_schedule": [-100, 0.001]
#         }),
#         ("RegionalPeriodicBracketTax", {
#             "region": "bottom",
#             "planner_id": "p_bottom",
#             "period": 100,
#             "bracket_spacing": "us-federal",
#             "usd_scaling": 1000,
#             "disable_taxes": False,
#             "tax_model": "model_wrapper",
#             "tax_annealing_schedule": [-100, 0.001]
#         }),
#     ],
#     "env_layout_file": "map_100x50_water_gaps_3percent_resources.txt",
#     "world_size": [100, 50],
#     "episode_length": 1000,
#     "starting_agent_coin": 10,
#     "fixed_four_skill_and_loc": False,
#     "n_agents": 4,
#     "planner_subclasses": ["TopPlanner", "BottomPlanner"],
#     "multi_action_mode_planner": True,
#     "multi_action_mode_agents": True,
#     "flatten_observations": True,
#     "flatten_masks": True,
#     "dense_log_frequency": 1
# }

# # =========================================================
# # Run Phase 2
# # =========================================================
# PHASE2_ITERS = 400

# ckpt_phase2_path = run_phase(
#     name="PHASE 2",
#     env_config_dict=env_config_dict_phase2,
#     policies_to_train=["p_top", "p_bottom"],
#     log_dir=PHASE2_DIR,
#     iters=PHASE2_ITERS,
#     restore_path=ckpt_phase1_path,
# )

# print("Done.")
# print("Phase 1 checkpoint:", ckpt_phase1_path)
# print("Phase 2 checkpoint:", ckpt_phase2_path)

# ray.shutdown()

In [ ]:
# import os
# import time
# import gc
# import ray

# from ray.rllib.agents.ppo import PPOTrainer
# from ray.tune.logger import UnifiedLogger
# from tutorials.rllib.env_wrapper import RLlibEnvWrapper

# # =========================================================
# # Ray init
# # =========================================================
# ray.init(ignore_reinit_error=True, log_to_driver=False)

# # =========================================================
# # Output dirs
# # =========================================================
# PHASE1_DIR = "phase1_tb"
# PHASE2_DIR = "phase2_tb"
# os.makedirs(PHASE1_DIR, exist_ok=True)
# os.makedirs(PHASE2_DIR, exist_ok=True)

# def make_logger_creator(base_dir):
#     def logger_creator(config):
#         ts = time.strftime("%Y-%m-%d_%H-%M-%S")
#         logdir = os.path.join(base_dir, f"run-{ts}")
#         os.makedirs(logdir, exist_ok=True)
#         return UnifiedLogger(config, logdir, loggers=None)
#     return logger_creator

# def policy_mapping_fun(agent_id):
#     aid = str(agent_id)
#     if aid.isdigit():
#         return "a"
#     if aid == "p_top":
#         return "p_top"
#     if aid == "p_bottom":
#         return "p_bottom"
#     return "a"

# # =========================================================
# # ENV CONFIGS
# # =========================================================
# env_config_dict_phase1 = {
#     "scenario_name": "custom/splitworld_overlay_regional",
#     "components": [
#         ('Build', {
#             'skill_dist': 'pareto',
#             'payment_max_skill_multiplier': 3,
#             'build_labor': 10,
#             'payment': 10
#         }),
#         ('ContinuousDoubleAuction', {
#             'max_bid_ask': 10,
#             'order_labor': 0.25,
#             'max_num_orders': 5,
#             'order_duration': 50
#         }),
#         ('Gather', {
#             'move_labor': 1,
#             'collect_labor': 1,
#             'skill_dist': 'pareto'
#         }),
#         ("RegionalPeriodicBracketTax", {
#             "region": "top",
#             "planner_id": "p_top",
#             "period": 100,
#             "bracket_spacing": "us-federal",
#             "usd_scaling": 1000,
#             "disable_taxes": True,
#         }),
#         ("RegionalPeriodicBracketTax", {
#             "region": "bottom",
#             "planner_id": "p_bottom",
#             "period": 100,
#             "bracket_spacing": "us-federal",
#             "usd_scaling": 1000,
#             "disable_taxes": True,
#         }),
#     ],
#     "env_layout_file": "map_100x50_water_gaps_3percent_resources.txt",
#     "world_size": [100, 50],
#     "episode_length": 1000,
#     "starting_agent_coin": 10,
#     "fixed_four_skill_and_loc": False,
#     "n_agents": 4,
#     "planner_subclasses": ["TopPlanner", "BottomPlanner"],
#     "multi_action_mode_planner": True,
#     "multi_action_mode_agents": True,
#     "flatten_observations": True,
#     "flatten_masks": True,
#     "dense_log_frequency": 1
# }

# env_config_dict_phase2 = {
#     "scenario_name": "custom/splitworld_overlay_regional",
#     "components": [
#         ('Build', {
#             'skill_dist': 'pareto',
#             'payment_max_skill_multiplier': 3,
#             'build_labor': 10,
#             'payment': 10
#         }),
#         ('ContinuousDoubleAuction', {
#             'max_bid_ask': 10,
#             'order_labor': 0.25,
#             'max_num_orders': 5,
#             'order_duration': 50
#         }),
#         ('Gather', {
#             'move_labor': 1,
#             'collect_labor': 1,
#             'skill_dist': 'pareto'
#         }),
#         ("RegionalPeriodicBracketTax", {
#             "region": "top",
#             "planner_id": "p_top",
#             "period": 100,
#             "bracket_spacing": "us-federal",
#             "usd_scaling": 1000,
#             "disable_taxes": False,
#             "tax_model": "model_wrapper",
#             "tax_annealing_schedule": [-100, 0.001]
#         }),
#         ("RegionalPeriodicBracketTax", {
#             "region": "bottom",
#             "planner_id": "p_bottom",
#             "period": 100,
#             "bracket_spacing": "us-federal",
#             "usd_scaling": 1000,
#             "disable_taxes": False,
#             "tax_model": "model_wrapper",
#             "tax_annealing_schedule": [-100, 0.001]
#         }),
#     ],
#     "env_layout_file": "map_100x50_water_gaps_3percent_resources.txt",
#     "world_size": [100, 50],
#     "episode_length": 1000,
#     "starting_agent_coin": 10,
#     "fixed_four_skill_and_loc": False,
#     "n_agents": 4,
#     "planner_subclasses": ["TopPlanner", "BottomPlanner"],
#     "multi_action_mode_planner": True,
#     "multi_action_mode_agents": True,
#     "flatten_observations": True,
#     "flatten_masks": True,
#     "dense_log_frequency": 1
# }

# # =========================================================
# # Build policy dict from a specific env
# # =========================================================
# def build_policies(env_obj):
#     obs_space_a   = env_obj.observation_space
#     act_space_a   = env_obj.action_space
#     obs_space_top = env_obj.observation_space_pl["p_top"]
#     act_space_top = env_obj.action_space_pl["p_top"]
#     obs_space_bot = env_obj.observation_space_pl["p_bottom"]
#     act_space_bot = env_obj.action_space_pl["p_bottom"]

#     policies = {
#         "a": (
#             None, obs_space_a, act_space_a,
#             {"lr": 3e-4}
#         ),
#         "p_top": (
#             None, obs_space_top, act_space_top,
#             {"lr": 1e-4, "entropy_coeff": 0.02}   # <-- add this
#         ),
#         "p_bottom": (
#             None, obs_space_bot, act_space_bot,
#             {"lr": 1e-4, "entropy_coeff": 0.02}   # <-- add this
#         ),
#     }
#     return policies

# # =========================================================
# # Common safer config
# # =========================================================
# def build_trainer_config(env_config_dict, policies, policies_to_train):
#     return {
#         "env": RLlibEnvWrapper,
#         "env_config": {
#             "env_config_dict": env_config_dict,
#             "num_envs_per_worker": 1,
#         },
#         "multiagent": {
#             "policies": policies,
#             "policies_to_train": policies_to_train,
#             "policy_mapping_fn": policy_mapping_fun,
#         },
#         "num_workers": 0,
#         "num_envs_per_worker": 1,
#         "framework": "tf",
#         "num_gpus": 0,

#         "rollout_fragment_length": 50,
#         "batch_mode": "truncate_episodes",
#         "train_batch_size": 500,
#         "sgd_minibatch_size": 64,
#         "num_sgd_iter": 2,

#         "log_level": "WARN",
#     }

# def train_phase(trainer, name, log_dir, iters):
#     ckpt_path = None
#     start_time = time.time()

#     for i in range(iters):
#         result = trainer.train()

#         if i % 25 == 0:
#             elapsed_min = (time.time() - start_time) / 60.0
#             print(
#                 f"[{name}] Iter={i:04d}/{iters} "
#                 f"reward={result.get('episode_reward_mean')} "
#                 f"elapsed={elapsed_min:.1f} min"
#             )

#         if i % 100 == 0 and i > 0:
#             ckpt_path = trainer.save(log_dir)
#             print(f"[{name}] Saved checkpoint: {ckpt_path}")

#         if i % 10 == 0:
#             gc.collect()

#     ckpt_path = trainer.save(log_dir)
#     total_min = (time.time() - start_time) / 60.0
#     print(f"[{name}] Final checkpoint: {ckpt_path}")
#     print(f"[{name}] Finished in {total_min:.1f} minutes")
#     return ckpt_path

# # =========================================================
# # Phase 1 setup
# # =========================================================
# env_obj_phase1 = RLlibEnvWrapper({"env_config_dict": env_config_dict_phase1}, verbose=False)
# policies_phase1 = build_policies(env_obj_phase1)

# trainer_phase1 = PPOTrainer(
#     config=build_trainer_config(
#         env_config_dict_phase1,
#         policies_phase1,
#         policies_to_train=["a"]
#     ),
#     logger_creator=make_logger_creator(PHASE1_DIR),
# )

# PHASE1_ITERS = 200
# ckpt_phase1_path = train_phase(trainer_phase1, "PHASE 1", PHASE1_DIR, PHASE1_ITERS)

# # =========================================================
# # Save only agent weights from Phase 1
# # =========================================================
# agent_weights = trainer_phase1.get_policy("a").get_weights()

# # =========================================================
# # Phase 2 setup -- IMPORTANT: use PHASE 2 spaces
# # =========================================================
# env_obj_phase2 = RLlibEnvWrapper({"env_config_dict": env_config_dict_phase2}, verbose=False)
# policies_phase2 = build_policies(env_obj_phase2)

# trainer_phase2 = PPOTrainer(
#     config=build_trainer_config(
#         env_config_dict_phase2,
#         policies_phase2,
#         policies_to_train=["p_top", "p_bottom"]
#     ),
#     logger_creator=make_logger_creator(PHASE2_DIR),
# )

# # Copy only the trained agent weights into Phase 2
# trainer_phase2.get_policy("a").set_weights(agent_weights)
# print("[PHASE 2] Loaded agent weights from Phase 1 (planners initialized fresh)")

# # Phase 1 trainer no longer needed
# trainer_phase1.stop()
# del trainer_phase1
# gc.collect()

# # =========================================================
# # Optional annealing verification
# # =========================================================
# try:
#     for comp in trainer_phase2.workers.local_worker().env.env.components:
#         if "BracketTax" in comp.name:
#             print(
#                 "[PHASE 2] Verify-start",
#                 comp.name,
#                 "schedule:", getattr(comp, "tax_annealing_schedule", None),
#                 "annealed_max:", getattr(comp, "_annealed_rate_max", None)
#             )
# except Exception as e:
#     print("[PHASE 2] Could not inspect tax components:", e)

# # # =========================================================
# # # Phase 2 training
# # # =========================================================
# # PHASE2_ITERS = 250
# # ckpt_phase2_path = train_phase(trainer_phase2, "PHASE 2", PHASE2_DIR, PHASE2_ITERS)

# # print("Done.")
# # print("Phase 1 checkpoint:", ckpt_phase1_path)
# # print("Phase 2 checkpoint:", ckpt_phase2_path)

# # trainer_phase2.stop()
# # del trainer_phase2
# # gc.collect()
# # ray.shutdown()